# Pemodelan dan evaluasi

Notebook ini membandingkan baseline CatBoost dengan stack XGBoost-LightGBM dan logistic meta-model. Preprocessing, training, dan tuning memakai pipeline yang sama dengan inferensi. Resep `official-data-stack-v1` menggunakan CSV resmi.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
assert (ROOT / 'scripts').exists(), 'Jalankan dari folder proyek atau notebooks.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / 'data'
FIGURES = ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
SEED = 2026
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.0)
plt.rcParams['figure.dpi'] = 110

## 1. Memisahkan training, calibration, dan evaluation

Training 80% dipakai untuk belajar preprocessing dan model dasar. Calibration 10% dipakai untuk
memilih baseline, regularisasi logistic meta-model, dan offset keputusan kelas. Evaluation 10%
dipakai untuk mengukur hasil setelah semua pilihan dikunci. Seed tetap 2026 dan 2027.

In [2]:
from scripts.workflow import load_data, split_data, run_workflow
from scripts.preprocess import build_preprocessor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

X, y = load_data(DATA_DIR)
train_x, cal_x, eval_x, train_y, cal_y, eval_y = split_data(X, y)
parts = {'train': train_y, 'calibration': cal_y, 'evaluation': eval_y}
display(pd.DataFrame({name: part.value_counts(normalize=True).sort_index() for name, part in parts.items()}).round(4))
print({name: len(part) for name, part in parts.items()})
assert set(train_x.index).isdisjoint(cal_x.index)
assert set(train_x.index).isdisjoint(eval_x.index)
assert set(cal_x.index).isdisjoint(eval_x.index)
preprocess = build_preprocessor()
assert isinstance(preprocess, Pipeline)
assert isinstance(preprocess.named_steps['columns'], ColumnTransformer)
print(preprocess)

,train,calibration,evaluation
damage_grade,,,
1,0.1034,0.1034,0.1034
2,0.1145,0.1145,0.1145
3,0.1790,0.1790,0.1790
4,0.2412,0.2412,0.2412
5,0.3619,0.3618,0.3619


{'train': 426772, 'calibration': 53346, 'evaluation': 53347}
Pipeline(steps=[('features', BuildingFeatures()),
                ('statistics', CrossFitStatistics()),
                ('columns',
                 ColumnTransformer(transformers=[('numeric',
                                                  SimpleImputer(keep_empty_features=True,
                                                                strategy='median'),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000023D8BC59010>),
                                                 ('categorical',
                                                  NativeCategories(),
                                                  ['district_id', 'vdcmun_id',
                                                   'ward_id',
                                                   'land_surface_condition',
                                                   'foundation_type',
         

## 2. Baseline, tuning terbatas, dan stack

Baseline membandingkan CatBoost depth 5/200 iterasi dan depth 6/300 iterasi pada calibration.
Tidak dilakukan early stopping atau pemilihan parameter pada evaluation.

Stack memakai tiga konfigurasi XGBoost dan tiga pasangan LightGBM multiclass/OvA. Setiap pasangan
LightGBM memakai blend tetap 60/40. Fitur group target dan tetangga ID pada training dihitung
secara 5-fold cross-fit. Calibration dan evaluation hanya ditransformasi dari statistik training.

Logistic meta-model mempelajari enam blok log-probabilitas pada calibration. Nilai C dipilih dari
0,03; 0,3; dan 3 menggunakan CV 5-fold calibration, lalu offset kelas dipilih dari probabilitas OOF.
OOF ini milik tingkat meta-model, bukan klaim bahwa seluruh stack memakai OOF training yang sama.
Evaluation tidak diberikan kepada fungsi training stack.

In [ ]:
# True menjalankan ulang seluruh workflow dari CSV mentah, tanpa cache eksperimen lama.
# Ubah ke False hanya jika ingin membaca hasil run yang sudah tersimpan.
RUN_TRAINING = True
report_path = ROOT / 'reports' / 'workflow_validation.json'
if RUN_TRAINING:
    report = run_workflow(DATA_DIR, ROOT)
else:
    assert report_path.exists(), 'Jalankan training terlebih dahulu.'
    report = json.loads(report_path.read_text(encoding='utf-8'))
print('Resep:', report['recipe'])
print('Versi dependensi:', report['versions'])
display(pd.DataFrame(report['baseline_search']))
display(pd.DataFrame(report['meta_search']))

## 3. Hasil pada evaluation

Sel berikut menampilkan Macro F1, classification report, dan confusion matrix berdasarkan hasil training.

In [ ]:
summary = pd.DataFrame([
    {'model': 'CatBoost baseline terpilih', 'macro_f1': report['baseline']['macro_f1']},
    {'model': 'Stack data resmi', 'macro_f1': report['stack']['macro_f1']},
])
display(summary.round(6))
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=summary, x='model', y='macro_f1', ax=ax, color='steelblue')
ax.set(ylim=(0, 1), ylabel='Macro F1', xlabel='', title='Evaluasi pada holdout yang sama')
plt.tight_layout()
fig.savefig(FIGURES / 'model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
per_class = pd.DataFrame(report['stack']['classification_report']).T
display(per_class.round(4))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
labels = [f'Grade {i}' for i in range(1, 6)]
for ax, name in zip(axes, ['baseline', 'stack']):
    matrix = np.asarray(report[name]['confusion_matrix'])
    normalized = matrix / matrix.sum(axis=1, keepdims=True)
    sns.heatmap(normalized, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set(title=f'Confusion matrix {name} (proporsi per kelas)', xlabel='Prediksi', ylabel='Aktual')
plt.tight_layout()
fig.savefig(FIGURES / 'confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
matrix = np.asarray(report['stack']['confusion_matrix'])
for grade in [2, 3, 4]:
    row = matrix[grade - 1].copy()
    row[grade - 1] = 0
    most_confused = row.argmax() + 1
    print(f'Grade {grade}: F1={per_class.loc[str(grade), "f1-score"]:.3f}; '
          f'kesalahan terbanyak ke Grade {most_confused} ({row.max():,} bangunan).')
print('Selisih Macro F1 stack terhadap baseline:',
      round(report['stack']['macro_f1'] - report['baseline']['macro_f1'], 6))

Diagonal confusion matrix menunjukkan prediksi yang benar. Nilai di luar diagonal membantu melihat
kelas yang saling tertukar. Analisis Grade 2–4 di atas menunjukkan kesalahan aktual pada run ini;
kelas tersebut tidak diasumsikan paling sulit tanpa melihat hasilnya.

Perbandingan baseline dengan stack membandingkan keseluruhan resep: fitur dan estimator berbeda.
Karena itu, selisih skor tidak dapat dianggap sebagai efek stacking saja.

## 4. Perbandingan OvA dan OvO yang sebanding

Kedua strategi menggunakan SGDClassifier `loss='log_loss'`, preprocessing, seed, dan 30.000 baris
training stratified yang sama. Evaluasi memakai seluruh evaluation holdout yang sama.
Subset dipakai hanya untuk ilustrasi strategi; baseline dan stack di atas memakai seluruh bagian training.
OvA melatih 5 classifier biner; OvO lengkap melatih 10 pasangan. Koreksi pasangan 2/3 dan 3/4 pada
eksperimen lama merupakan koreksi parsial, bukan OvO lengkap.

In [ ]:
strategy_rows = [
    {'strategi': name, 'macro_f1': result['macro_f1'],
     'baris_training': result['training_rows'], 'baris_evaluation': result['evaluation_rows'],
     'classifier_biner': result['binary_classifiers']}
    for name, result in report['strategies'].items()
]
display(pd.DataFrame(strategy_rows).round(6))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, result) in zip(axes, report['strategies'].items()):
    matrix = np.asarray(result['confusion_matrix'])
    sns.heatmap(matrix / matrix.sum(axis=1, keepdims=True), annot=True, fmt='.2f',
                cmap='Blues', vmin=0, vmax=1, xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set(title=name, xlabel='Prediksi', ylabel='Aktual')
plt.tight_layout()
fig.savefig(FIGURES / 'ova_ovo_comparison.png', bbox_inches='tight')
plt.show()

Perbandingan OvA dan OvO menggunakan model dasar, fitur, serta pembagian data yang sama.

## 5. Model tersimpan

Workflow menyimpan preprocessing, model dasar, logistic meta-model, offset kelas, konfigurasi, dan versi dependensi. Model dasar menggunakan training 80%, sedangkan meta-model dan kalibrasi menggunakan calibration. Model disimpan secara lokal pada folder `models/` dan diabaikan Git.

In [ ]:
import joblib
model_path = ROOT / 'models' / 'reproducible_stack.joblib'
model = joblib.load(model_path)
assert report['serialization_predictions_equal']
display(pd.DataFrame({'kelas': labels, 'offset_logit': model.bias_}))
print('Model tersimpan:', model_path.relative_to(ROOT))
print('Ukuran model (MB):', round(model_path.stat().st_size / 1_000_000, 2))
print('Lanjutkan ke notebook 03 untuk membuat kandidat submission terpisah.')